# Pandalyzer — Manual Connector Tests
## Purpose
Test CSV loading, CSV export, JSON quality-report export, directory creation, and round-trip reading. Outputs are written to a temporary test folder, not over the source dataset.

## 1. Imports and paths

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd

_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in _candidates if (p / "database").is_dir() and (p / "src" / "autoclean").is_dir()),
    Path("/mnt/data/pandalyzer_manual_tests"),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from autoclean.cleaner import remove_duplicate_rows, handle_missing_values, clean_dataframe
from autoclean.profiler import profile_dataframe, compare_data_quality
from autoclean.validators import validate_missing_values, validate_required_columns, validate_column_types, validate_numeric_ranges
from autoclean.connectors import load_csv, save_csv, save_quality_report

csv_path = PROJECT_ROOT / "database" / "messy_retail_orders.csv"
print("Project root:", PROJECT_ROOT)
print("Dataset available:", csv_path.exists())

OUTPUT_DIR = PROJECT_ROOT / "manual_test_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Project root: /run/media/amateur/MATRIX/Pandalyzer/AutoClean
Dataset available: True


## 2. Test CSV loading
Load the messy retail dataset if available; otherwise use a deterministic synthetic DataFrame.

In [2]:
if csv_path.exists():
    loaded_df = load_csv(csv_path)
    print("Loaded:", csv_path.name, "| shape:", loaded_df.shape)
    display(loaded_df.head())
    assert len(loaded_df.columns) > 0
else:
    loaded_df = pd.DataFrame({"id":[1,2], "amount":[10.0,20.0]})
    print("CSV unavailable; using synthetic data.")

Loaded: messy_retail_orders.csv | shape: (10000, 20)


,order_id,order_date,customer_id,product_id,category,quantity,unit_price,discount_pct,order_amount,payment_method,order_status,city,state,sales_channel,customer_email,delivery_days,rating,coupon_code,is_returned,notes
0,ORD009826,2024-06-19 16:56:11,CUST01648,NaN,Grocery,5.0,96.55,30.0,NaN,Credit Card,Cancelled,Mumbai,Maharashtra,Website,not-an-email,12.0,1.0,NaN,No,NaN
1,ORD004017,2025-03-18 10:42:56,CUST01249,\tPRD0244\t,Home & Kitchen,NaN,1567.44,5.0,4467.2,COD,\tShipped\t,Hyderabad,Telangana,Website,customer1175@example.com,NaN,3.0,NaN,No,NaN
2,ORD008732,2026-09-30 12:23:18,CUST00211,PRD0160,Clothing,2.0,958.08,0.0,1916.16,Credit Card,Returned,lucknow,Uttar Pradesh,Store,NaN,11.0,1.0,NaN,No,Gift order
3,ORD009698,2024-01-18 19:02:44,CUST00406,PRD0178,Electronics,2.0,421.03,0.0,NaN,Wallet,Returned,Bengaluru,NaN,NaN,@example.com,9.0,2.0,FESTIVE25,NaN,Customer requested update
4,ORD004226,NaN,CUST00610,PRD0235,Sports,2.0,1653.18,20.0,2645.09,Cash,Cancelled,Delhi,Delhi,Website,\tcustomer780@example.com \t,8.0,5.0,NaN,NaN,Gift order


## 3. Test CSV export and round-trip
Export a small DataFrame and reload it to verify values are preserved.

In [3]:
sample = pd.DataFrame({"id":[1,2,3], "category":["books","home","sports"], "amount":[10.5,20.0,30.25]})
csv_out = save_csv(sample, OUTPUT_DIR / "connector_roundtrip.csv")
roundtrip = load_csv(csv_out)
display(roundtrip)
pd.testing.assert_frame_equal(roundtrip, sample, check_dtype=False)
assert csv_out.exists()
print("PASS: CSV round-trip")

,id,category,amount
0,1,books,10.50
1,2,home,20.00
2,3,sports,30.25


PASS: CSV round-trip


## 4. Test parent-directory creation
Saving into a nested, not-yet-existing folder should create its parent directories.

In [4]:
nested_out = save_csv(sample, OUTPUT_DIR / "nested" / "deep" / "sample.csv")
assert nested_out.exists()
assert load_csv(nested_out).shape == sample.shape
print("PASS: parent directory creation")

PASS: parent directory creation


## 5. Test quality-report JSON export
Export a report containing ordinary nested data and pandas objects, then parse it back.

In [5]:
report = {
    "summary": {"rows": len(sample), "missing_cells": int(sample.isna().sum().sum())},
    "columns": sample.describe(include="all"),
    "missing_by_column": sample.isna().sum(),
    "notes": ["manual connector test", "round-trip"],
}
json_out = save_quality_report(report, OUTPUT_DIR / "quality_report.json")
with json_out.open(encoding="utf-8") as f:
    loaded_report = json.load(f)
print("JSON saved:", json_out)
display(pd.Series(loaded_report["summary"]).to_frame("value"))
assert loaded_report["summary"]["rows"] == 3
assert loaded_report["summary"]["missing_cells"] == 0
assert "columns" in loaded_report and "missing_by_column" in loaded_report
print("PASS: quality report JSON export")

JSON saved: /run/media/amateur/MATRIX/Pandalyzer/AutoClean/manual_test_outputs/quality_report.json


,value
rows,3
missing_cells,0


PASS: quality report JSON export


## 6. Test invalid CSV export input
The connector should reject non-DataFrame input with a TypeError.

In [6]:
try:
    save_csv([1,2,3], OUTPUT_DIR / "invalid.csv")
except TypeError:
    print("PASS: invalid input rejected")
else:
    raise AssertionError("save_csv should reject non-DataFrame input")

PASS: invalid input rejected
